In [1]:
!nvidia-smi

Wed Mar 18 06:26:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


# Creating a Shared Library and Using it in Python

## Code for Matrix Multiplication (Adding to Shared Library)

In [3]:
%%writefile matrix_lib.cu
#include <cuda_runtime.h>
#include <stdio.h>
#define TILE_WIDTH 16

__global__ void matrixMultiplyTiled(float *A, float *B, float *C, int N) {
    __shared__ float ds_A[TILE_WIDTH][TILE_WIDTH];
    __shared__ float ds_B[TILE_WIDTH][TILE_WIDTH];

    int tx = threadIdx.x;
    int ty = threadIdx.y;

    int Row = blockIdx.y * TILE_WIDTH + ty;
    int Col = blockIdx.x * TILE_WIDTH + tx;

    float Pvalue = 0.0f;

    int numTiles = (N + TILE_WIDTH - 1) / TILE_WIDTH;
    for (int m = 0; m < numTiles; ++m) {
        int aCol = m * TILE_WIDTH + tx;
        if (Row < N && aCol < N) {
            ds_A[ty][tx] = A[Row * N + aCol];
        } else {
            ds_A[ty][tx] = 0.0f;
        }

        int bRow = m * TILE_WIDTH + ty;
        if (Col < N && bRow < N) {
            ds_B[ty][tx] = B[bRow * N + Col];
        } else {
            ds_B[ty][tx] = 0.0f;
        }

        __syncthreads();

        for (int k = 0; k < TILE_WIDTH; ++k) {
            Pvalue += ds_A[ty][k] * ds_B[k][tx];
        }

        __syncthreads();
    }

    if (Row < N && Col < N) {
        C[Row * N + Col] = Pvalue;
    }
}

extern "C" void gpu_matrix_multiply(float *h_A, float *h_B, float *h_C, int N) {
    size_t size = N * N * sizeof(float);

    float *d_A;
    float *d_B;
    float *d_C;

    cudaMalloc((void**)&d_A, size);
    cudaMalloc((void**)&d_B, size);
    cudaMalloc((void**)&d_C, size);

    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    dim3 block(TILE_WIDTH, TILE_WIDTH);

    int numBlocksX = (N + TILE_WIDTH - 1) / TILE_WIDTH;
    int numBlocksY = (N + TILE_WIDTH - 1) / TILE_WIDTH;
    dim3 grid(numBlocksX, numBlocksY);

    matrixMultiplyTiled<<<grid, block>>>(d_A, d_B, d_C, N);
    cudaDeviceSynchronize();
    cudaMemcpy(h_C, d_C, size, cudaMemcpyDeviceToHost);

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);
}

Writing matrix_lib.cu


## Code for Convolution Function (Adding to Shared Library)

In [4]:
%%writefile conv_lib.cu
#include <cuda_runtime.h>
#include <stdio.h>

__global__ void convolution2D_GPU(
    unsigned char *image,
    int *kernel,
    int *output,
    int width,
    int height,
    int K
) {
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;

    if (x >= width || y >= height) return;

    int half = K / 2;
    int sum = 0;

    for (int ky = -half; ky <= half; ky++) {
        for (int kx = -half; kx <= half; kx++) {
            int ix = x + kx;
            int iy = y + ky;

            if (ix >= 0 && ix < width && iy >= 0 && iy < height) {
                int pixel = image[iy * width + ix];
                int weight = kernel[(ky + half) * K + (kx + half)];
                sum += pixel * weight;
            }
        }
    }

    output[y * width + x] = sum;
}

extern "C" void gpu_convolution(
    unsigned char *h_image,
    int *h_kernel,
    int *h_output,
    int width,
    int height,
    int K
) {
    unsigned char *d_image;
    int *d_kernel;
    int *d_output;

    size_t img_size = width * height * sizeof(unsigned char);
    size_t kernel_size = K * K * sizeof(int);
    size_t out_size = width * height * sizeof(int);

    cudaMalloc(&d_image, img_size);
    cudaMalloc(&d_kernel, kernel_size);
    cudaMalloc(&d_output, out_size);

    cudaMemcpy(d_image, h_image, img_size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_kernel, h_kernel, kernel_size, cudaMemcpyHostToDevice);

    dim3 threads(16, 16);
    dim3 blocks(
        (width + 15) / 16,
        (height + 15) / 16
    );

    convolution2D_GPU<<<blocks, threads>>>(
        d_image, d_kernel, d_output,
        width, height, K
    );
    cudaDeviceSynchronize();

    cudaMemcpy(h_output, d_output, out_size, cudaMemcpyDeviceToHost);

    cudaFree(d_image);
    cudaFree(d_kernel);
    cudaFree(d_output);
}


Writing conv_lib.cu


In [5]:
!nvcc -Xcompiler -fPIC -shared matrix_lib.cu conv_lib.cu -o libmatrix.so

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [6]:
!nm -D libmatrix.so | grep gpu_convolution

00000000000090cc T gpu_convolution


## Testing Matrix Multiplication Through Python

In [7]:
import ctypes
import numpy as np
import time
import sys
import os

lib = ctypes.cdll.LoadLibrary("./libmatrix.so")
lib.gpu_matrix_multiply.argtypes = [
    np.ctypeslib.ndpointer(dtype=np.float32, ndim=1, flags="C_CONTIGUOUS"),
    np.ctypeslib.ndpointer(dtype=np.float32, ndim=1, flags="C_CONTIGUOUS"),
    np.ctypeslib.ndpointer(dtype=np.float32, ndim=1, flags="C_CONTIGUOUS"),
    ctypes.c_int
]

N = 512

for i in range(5):
    print(f"\nTesting with matrix size N = {N}")

    A = np.random.rand(N, N).astype(np.float32)
    B = np.random.rand(N, N).astype(np.float32)
    C = np.zeros((N, N), dtype=np.float32)

    start = time.time()
    lib.gpu_matrix_multiply(A.ravel(), B.ravel(), C.ravel(), N)
    end = time.time()
    elapsed_time = end - start

    print(f"CUDA library time (N={N}): {elapsed_time * 1000:.4f} ms")

    N = N * 2


Testing with matrix size N = 512
CUDA library time (N=512): 473.8009 ms

Testing with matrix size N = 1024
CUDA library time (N=1024): 7.8731 ms

Testing with matrix size N = 2048
CUDA library time (N=2048): 56.2027 ms

Testing with matrix size N = 4096
CUDA library time (N=4096): 324.6248 ms

Testing with matrix size N = 8192
CUDA library time (N=8192): 2040.1447 ms


## Testing Convolution Function Through Python


In [8]:
import ctypes
import numpy as np
import time
import os

# Load shared library
lib_path = os.path.abspath("./libmatrix.so")
lib = ctypes.CDLL(lib_path)

# Bind function
gpu_convolution = lib.gpu_convolution
gpu_convolution.argtypes = [
    np.ctypeslib.ndpointer(dtype=np.uint8, ndim=1, flags="C_CONTIGUOUS"),
    np.ctypeslib.ndpointer(dtype=np.int32, ndim=1, flags="C_CONTIGUOUS"),
    np.ctypeslib.ndpointer(dtype=np.int32, ndim=1, flags="C_CONTIGUOUS"),
    ctypes.c_int,  # width
    ctypes.c_int,  # height
    ctypes.c_int   # kernel size K
]
gpu_convolution.restype = None

# Load CUDA runtime for synchronization
cudart = ctypes.CDLL("libcudart.so")
cudart.cudaDeviceSynchronize.restype = ctypes.c_int

# Define convolution kernels
kernels = {
    "edge_3x3": np.array([
        -1, -1, -1,
        -1,  8, -1,
        -1, -1, -1
    ], dtype=np.int32),

    "sobel_x_3x3": np.array([
        -1,  0,  1,
        -2,  0,  2,
        -1,  0,  1
    ], dtype=np.int32),

    "gaussian_5x5": np.array([
        1,  4,  6,  4,  1,
        4, 16, 24, 16,  4,
        6, 24, 36, 24,  6,
        4, 16, 24, 16,  4,
        1,  4,  6,  4,  1
    ], dtype=np.int32),

    "edge_7x7": np.array([
        -1, -1, -1, -1, -1, -1, -1,
        -1, -1, -1, -1, -1, -1, -1,
        -1, -1, -1, -1, -1, -1, -1,
        -1, -1, -1, 48, -1, -1, -1,
        -1, -1, -1, -1, -1, -1, -1,
        -1, -1, -1, -1, -1, -1, -1,
        -1, -1, -1, -1, -1, -1, -1
    ], dtype=np.int32)
}

# Matrix sizes
image_sizes = [512, 1024, 2048, 4096, 8192]

print("CUDA Convolution Performance Results\n")

for size in image_sizes:
    width = height = size
    print(f"Image size: {width} x {height}")

    # Generate random grayscale image
    image = np.random.randint(0, 256, size=(height, width), dtype=np.uint8)
    image_flat = image.ravel()

    for kernel_name, kernel in kernels.items():
        kernel = np.ascontiguousarray(kernel, dtype=np.int32)
        N = int(np.sqrt(kernel.size))

        output = np.zeros((height, width), dtype=np.int32)
        output_flat = output.ravel()

        # Warm-up run
        gpu_convolution(image_flat, kernel, output_flat, width, height, N)
        cudart.cudaDeviceSynchronize()

        # Timed run
        start = time.time()

        gpu_convolution(image_flat, kernel, output_flat, width, height, N)
        cudart.cudaDeviceSynchronize()

        end = time.time()
        elapsed_ms = (end - start) * 1000.0

        print(f"  Kernel: {kernel_name:<15} (N={N}x{N}) -> {elapsed_ms:.4f} ms")

    print("-" * 60)

CUDA Convolution Performance Results

Image size: 512 x 512
  Kernel: edge_3x3        (N=3x3) -> 0.8113 ms
  Kernel: sobel_x_3x3     (N=3x3) -> 0.6273 ms
  Kernel: gaussian_5x5    (N=5x5) -> 0.7498 ms
  Kernel: edge_7x7        (N=7x7) -> 0.6595 ms
------------------------------------------------------------
Image size: 1024 x 1024
  Kernel: edge_3x3        (N=3x3) -> 1.7054 ms
  Kernel: sobel_x_3x3     (N=3x3) -> 1.8463 ms
  Kernel: gaussian_5x5    (N=5x5) -> 1.9052 ms
  Kernel: edge_7x7        (N=7x7) -> 1.9710 ms
------------------------------------------------------------
Image size: 2048 x 2048
  Kernel: edge_3x3        (N=3x3) -> 6.3305 ms
  Kernel: sobel_x_3x3     (N=3x3) -> 5.2295 ms
  Kernel: gaussian_5x5    (N=5x5) -> 5.6157 ms
  Kernel: edge_7x7        (N=7x7) -> 6.3586 ms
------------------------------------------------------------
Image size: 4096 x 4096
  Kernel: edge_3x3        (N=3x3) -> 21.0991 ms
  Kernel: sobel_x_3x3     (N=3x3) -> 21.3385 ms
  Kernel: gaussian_5x5   

## Testing Different Convolution Filter Matrices (Not Through Shared Library)

In [9]:
%%writefile conv3x3_gpu.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

__global__ void convolution2D_GPU(
    unsigned char *image,
    int *kernel,
    int *output,
    int width,
    int height
) {
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;

    if (x >= width || y >= height) return;

    int sum = 0;
    int k = 1;  // 3x3 kernel radius

    for (int ky = -k; ky <= k; ky++) {
        for (int kx = -k; kx <= k; kx++) {
            int ix = x + kx;
            int iy = y + ky;
            if (ix >= 0 && ix < width && iy >= 0 && iy < height) {
                int pixel = image[iy * width + ix];
                int weight = kernel[(ky + k) * 3 + (kx + k)];
                sum += pixel * weight;
            }
        }
    }
    output[y * width + x] = sum;
}

int main(int argc, char **argv) {
    int M = (argc > 1) ? atoi(argv[1]) : 512;
    int width = M;
    int height = M;

    size_t img_size = width * height * sizeof(unsigned char);
    size_t out_size = width * height * sizeof(int);

    unsigned char *h_img = (unsigned char*)malloc(img_size);
    int *h_out = (int*)malloc(out_size);

    int h_kernel[9] = {
        -1, -1, -1,
        -1,  8, -1,
        -1, -1, -1
    };

    for (int i = 0; i < width * height; i++)
        h_img[i] = rand() % 256;

    unsigned char *d_img;
    int *d_kernel, *d_out;

    cudaMalloc(&d_img, img_size);
    cudaMalloc(&d_kernel, 9 * sizeof(int));
    cudaMalloc(&d_out, out_size);

    cudaMemcpy(d_img, h_img, img_size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_kernel, h_kernel, 9 * sizeof(int), cudaMemcpyHostToDevice);

    dim3 block(16, 16);
    dim3 grid((width + 15) / 16, (height + 15) / 16);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);
    convolution2D_GPU<<<grid, block>>>(d_img, d_kernel, d_out, width, height);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float ms;
    cudaEventElapsedTime(&ms, start, stop);
    printf("CUDA 3x3 convolution time (M=%d): %.4f ms\n", M, ms);

    cudaMemcpy(h_out, d_out, out_size, cudaMemcpyDeviceToHost);

    cudaFree(d_img);
    cudaFree(d_kernel);
    cudaFree(d_out);
    free(h_img);
    free(h_out);

    return 0;
}


Writing conv3x3_gpu.cu


In [10]:
%%writefile conv5x5_gpu.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define KERNEL_SIZE 5
#define KERNEL_RADIUS 2

__global__ void convolution2D_GPU(
    unsigned char *image,
    int *kernel,
    int *output,
    int width,
    int height
) {
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;

    if (x >= width || y >= height) return;

    int sum = 0;

    for (int ky = -KERNEL_RADIUS; ky <= KERNEL_RADIUS; ky++) {
        for (int kx = -KERNEL_RADIUS; kx <= KERNEL_RADIUS; kx++) {
            int ix = x + kx;
            int iy = y + ky;

            if (ix >= 0 && ix < width && iy >= 0 && iy < height) {
                int pixel = image[iy * width + ix];
                int weight = kernel[
                    (ky + KERNEL_RADIUS) * KERNEL_SIZE +
                    (kx + KERNEL_RADIUS)
                ];
                sum += pixel * weight;
            }
        }
    }

    output[y * width + x] = sum;
}

int main(int argc, char **argv) {
    int M = (argc > 1) ? atoi(argv[1]) : 512;
    int width = M;
    int height = M;

    size_t img_size = width * height * sizeof(unsigned char);
    size_t out_size = width * height * sizeof(int);
    size_t ker_size = KERNEL_SIZE * KERNEL_SIZE * sizeof(int);

    unsigned char *h_img = (unsigned char*)malloc(img_size);
    int *h_out = (int*)malloc(out_size);

    int h_kernel_5x5[KERNEL_SIZE * KERNEL_SIZE] = {
        -1, -1, -1, -1, -1,
        -1, -1, -1, -1, -1,
        -1, -1, 24, -1, -1,
        -1, -1, -1, -1, -1,
        -1, -1, -1, -1, -1
    };

    for (int i = 0; i < width * height; i++) {
        h_img[i] = rand() % 256;
    }

    unsigned char *d_img;
    int *d_kernel, *d_out;

    cudaMalloc(&d_img, img_size);
    cudaMalloc(&d_kernel, ker_size);
    cudaMalloc(&d_out, out_size);

    cudaMemcpy(d_img, h_img, img_size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_kernel, h_kernel_5x5, ker_size, cudaMemcpyHostToDevice);

    dim3 block(16, 16);
    dim3 grid((width + 15) / 16, (height + 15) / 16);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);
    convolution2D_GPU<<<grid, block>>>(d_img, d_kernel, d_out, width, height);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float ms = 0.0f;
    cudaEventElapsedTime(&ms, start, stop);
    printf("CUDA 5x5 convolution time (M=%d): %.4f ms\n", M, ms);

    cudaMemcpy(h_out, d_out, out_size, cudaMemcpyDeviceToHost);

    cudaFree(d_img);
    cudaFree(d_kernel);
    cudaFree(d_out);
    free(h_img);
    free(h_out);

    return 0;
}


Writing conv5x5_gpu.cu


In [11]:
%%writefile conv7x7_gpu.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define KERNEL_SIZE 7
#define KERNEL_RADIUS 3

__global__ void convolution2D_GPU(
    unsigned char *image,
    int *kernel,
    int *output,
    int width,
    int height
) {
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;

    if (x >= width || y >= height) return;

    int sum = 0;

    for (int ky = -KERNEL_RADIUS; ky <= KERNEL_RADIUS; ky++) {
        for (int kx = -KERNEL_RADIUS; kx <= KERNEL_RADIUS; kx++) {
            int ix = x + kx;
            int iy = y + ky;

            if (ix >= 0 && ix < width && iy >= 0 && iy < height) {
                int pixel = image[iy * width + ix];
                int weight =
                    kernel[(ky + KERNEL_RADIUS) * KERNEL_SIZE +
                           (kx + KERNEL_RADIUS)];
                sum += pixel * weight;
            }
        }
    }

    output[y * width + x] = sum;
}

int main(int argc, char **argv) {

    int M = (argc > 1) ? atoi(argv[1]) : 512;
    int width = M;
    int height = M;

    size_t img_size = width * height * sizeof(unsigned char);
    size_t out_size = width * height * sizeof(int);
    size_t ker_size = KERNEL_SIZE * KERNEL_SIZE * sizeof(int);

    unsigned char *h_img = (unsigned char*)malloc(img_size);
    int *h_out = (int*)malloc(out_size);

    int h_kernel_7x7[KERNEL_SIZE * KERNEL_SIZE] = {
        -1, -1, -1, -1, -1, -1, -1,
        -1, -1, -1, -1, -1, -1, -1,
        -1, -1, -1, -1, -1, -1, -1,
        -1, -1, -1, 48, -1, -1, -1,
        -1, -1, -1, -1, -1, -1, -1,
        -1, -1, -1, -1, -1, -1, -1,
        -1, -1, -1, -1, -1, -1, -1
    };

    for (int i = 0; i < width * height; i++) {
        h_img[i] = rand() % 256;
    }

    unsigned char *d_img;
    int *d_kernel, *d_out;

    cudaMalloc(&d_img, img_size);
    cudaMalloc(&d_kernel, ker_size);
    cudaMalloc(&d_out, out_size);

    cudaMemcpy(d_img, h_img, img_size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_kernel, h_kernel_7x7, ker_size, cudaMemcpyHostToDevice);

    dim3 block(16, 16);
    dim3 grid(
        (width + block.x - 1) / block.x,
        (height + block.y - 1) / block.y
    );

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);
    convolution2D_GPU<<<grid, block>>>(d_img, d_kernel, d_out, width, height);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float ms = 0.0f;
    cudaEventElapsedTime(&ms, start, stop);
    printf("CUDA 7x7 convolution time (M=%d): %.4f ms\n", M, ms);

    cudaMemcpy(h_out, d_out, out_size, cudaMemcpyDeviceToHost);

    cudaFree(d_img);
    cudaFree(d_kernel);
    cudaFree(d_out);
    free(h_img);
    free(h_out);

    return 0;
}

Writing conv7x7_gpu.cu


In [12]:
!nvcc -arch=sm_75 conv3x3_gpu.cu -o conv3x3_gpu
!nvcc -arch=sm_75 conv5x5_gpu.cu -o conv5x5_gpu
!nvcc -arch=sm_75 conv7x7_gpu.cu -o conv7x7_gpu

In [13]:
!./conv3x3_gpu 512
!./conv3x3_gpu 1024
!./conv3x3_gpu 2048
!./conv3x3_gpu 4096
!./conv3x3_gpu 8192

CUDA 3x3 convolution time (M=512): 0.1434 ms
CUDA 3x3 convolution time (M=1024): 0.1587 ms
CUDA 3x3 convolution time (M=2048): 0.3480 ms
CUDA 3x3 convolution time (M=4096): 0.8925 ms
CUDA 3x3 convolution time (M=8192): 4.3909 ms


In [14]:
!./conv5x5_gpu 512
!./conv5x5_gpu 1024
!./conv5x5_gpu 2048
!./conv5x5_gpu 4096
!./conv5x5_gpu 8192

CUDA 5x5 convolution time (M=512): 0.1407 ms
CUDA 5x5 convolution time (M=1024): 0.2595 ms
CUDA 5x5 convolution time (M=2048): 0.7252 ms
CUDA 5x5 convolution time (M=4096): 2.0076 ms
CUDA 5x5 convolution time (M=8192): 10.4727 ms


In [15]:
!./conv7x7_gpu 512
!./conv7x7_gpu 1024
!./conv7x7_gpu 2048
!./conv7x7_gpu 4096
!./conv7x7_gpu 8192

CUDA 7x7 convolution time (M=512): 0.1964 ms
CUDA 7x7 convolution time (M=1024): 0.3155 ms
CUDA 7x7 convolution time (M=2048): 1.0108 ms
CUDA 7x7 convolution time (M=4096): 3.6066 ms
CUDA 7x7 convolution time (M=8192): 27.8954 ms
